Pilot workflow: mount Drive, choose only a small SynthHuman download (`single-sample` or `single-chunk`), generate and validate the manifest, then train on a deterministic 500-sample subset for 5 epochs. Full-release acquisition lives in the separate `download_synthhuman_full.ipynb` notebook and is disabled by default; it is not part of the pilot workflow. The full download is about 330 GB plus extracted-data headroom, so check Drive capacity before enabling its download cell. Training shows per-batch progress and CUDA/GPU details. Dataset, pilot manifest, caches, checkpoints, metrics, and exports persist under Drive. This trial supervises depth and normals only; SynthHuman has no albedo target.


In [ ]:
# 0. Dependency preflight (run once per fresh Colab runtime)
import importlib.util
missing = [name for name in ('timm', 'skimage', 'cv2') if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing) + '. In a separate Colab cell run: %pip install timm scikit-image opencv-python-headless, then restart the runtime and rerun setup.')
print('Dependencies present: timm, scikit-image, and OpenCV (cv2; EXR-capable Colab build)')


In [ ]:
# 1. Runtime, imports, and experiment settings
# Run this after connecting the Colab runtime. Drive is mounted in the next cell.
import os, json, math, random, time, platform
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
try:
    from skimage.metrics import structural_similarity as skimage_ssim
except ImportError:
    skimage_ssim = None

SEED = 42
INPUT_SIZE = (256, 256)  # H, W; fixed shape for initial Core ML conversion
BATCH_SIZE, EPOCHS, NUM_WORKERS = 8, 5, 2
LR, WEIGHT_DECAY, VAL_FRACTION, TEST_FRACTION = 1e-4, 1e-4, 0.15, 0.15
DEPTH_MODE = 'metric'  # SynthHuman absolute z-depth converted from centimetres to metres
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = torch.cuda.is_available()
print({'device':str(DEVICE),'torch':torch.__version__,'python':platform.python_version(),'input_size':INPUT_SIZE})
if torch.cuda.is_available():
    print({'cuda_available':True,'cuda_version':torch.version.cuda,'cudnn_version':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),'gpu_count':torch.cuda.device_count(),'gpu_capability':torch.cuda.get_device_capability(0),'gpu_memory_GiB':round(torch.cuda.get_device_properties(0).total_memory/1024**3,2)})
else:
    print({'cuda_available':False,'message':'Training will run on CPU; in Colab choose Runtime > Change runtime type > GPU.'})

### Google Drive persistence
Colab runtimes are temporary. Mount Drive at the start of every new runtime session, then keep the dataset, manifest, pretrained-weight caches, checkpoints, training history, metrics, and Core ML exports under `MyDrive/AppleCapstoneRelighting`.

In [ ]:
# 2. Mount Google Drive and configure persistent paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
DATA_ROOT = DRIVE_ROOT / 'dataset'
MANIFEST = DATA_ROOT / 'manifest.jsonl'
OUTPUT_DIR = DRIVE_ROOT / 'runs' / 'fastvit_t8_fpn_pilot500_5ep_v1'
CACHE_ROOT = DRIVE_ROOT / 'cache'

# Keep downloaded model/dataset assets between Colab sessions as well.
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
for folder in (DATA_ROOT, OUTPUT_DIR, CACHE_ROOT / 'torch', CACHE_ROOT / 'huggingface'):
    folder.mkdir(parents=True, exist_ok=True)

assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive is not mounted.'
print('Persistent Drive root:', DRIVE_ROOT)
print('Dataset root:', DATA_ROOT)
print('Manifest:', MANIFEST)
print('Run outputs:', OUTPUT_DIR)
print('Run ID/config folder:', OUTPUT_DIR.name)
print('Drive folders are ready. Put paired RGB/depth/normal/albedo/mask assets and manifest in the dataset folder.')

### Download SynthHuman subset for the pilot (persistent on Drive)
This pipeline cell only downloads a smoke-test sample or a single chunk. Full-release downloading is in the separate `download_synthhuman_full.ipynb` notebook, so it cannot interrupt this pilot workflow. The official SynthHuman release provides RGB, absolute depth, normals, and alpha masks, but no albedo labels.


In [ ]:
# 2a. Download/confirm the small SynthHuman subset used by the pipeline
import subprocess, sys
DOWNLOAD_MODE = 'single-chunk'  # pipeline choices: 'single-sample', 'single-chunk'
DAVID_REPO = Path('/content/DAViD')
SYNTHHUMAN_PARENT = DATA_ROOT / 'sources'
SYNTHHUMAN_ROOT = SYNTHHUMAN_PARENT / 'SynthHuman'
if DOWNLOAD_MODE not in ('single-sample','single-chunk'):
    raise ValueError("Pipeline download cell only supports 'single-sample' or 'single-chunk'. Use `download_synthhuman_full.ipynb` for the optional full release.")
if not (DAVID_REPO / 'download_data.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/microsoft/DAViD.git', str(DAVID_REPO)], check=True)
existing_samples = list(SYNTHUMAN_ROOT.glob('rgb_*.png')) if SYNTHUMAN_ROOT.exists() else []
required_count = 1 if DOWNLOAD_MODE == 'single-sample' else 5000
if len(existing_samples) < required_count:
    flag = '--single-sample' if DOWNLOAD_MODE == 'single-sample' else '--single-chunk'
    subprocess.run([sys.executable, str(DAVID_REPO / 'download_data.py'), str(SYNTHHUMAN_PARENT), flag], check=True)
else:
    print(f'Existing Drive data found ({len(existing_samples)} RGB files); skipping pipeline subset download.')
print('SynthHuman data path:', SYNTHHUMAN_ROOT)
print('RGB samples available:', len(list(SYNTHUMAN_ROOT.glob('rgb_*.png'))))

In [ ]:
# 3. Verify downloaded files and create the JSONL manifest automatically
import re
SYNTHHUMAN_ROOT = DATA_ROOT / 'sources' / 'SynthHuman'
RGB_FILES = sorted(SYNTHHUMAN_ROOT.glob('rgb_*.png'))
print('SynthHuman RGB sample count:', len(RGB_FILES))
if not RGB_FILES:
    raise FileNotFoundError('No SynthHuman samples found. Run the preceding download cell first.')
manifest_rows=[]
incomplete_samples=[]
for image_path in RGB_FILES:
    sample_id=re.search(r'rgb_(\d+)\.png$', image_path.name).group(1)
    depth_path=SYNTHHUMAN_ROOT/f'depth_{sample_id}.exr'
    normal_path=SYNTHHUMAN_ROOT/f'normal_{sample_id}.exr'
    mask_path=SYNTHHUMAN_ROOT/f'alpha_{sample_id}.png'
    missing=[str(q.name) for q in (depth_path,normal_path,mask_path) if not q.is_file()]
    if missing:
        incomplete_samples.append({'sample_id':sample_id,'missing':missing}); continue
    manifest_rows.append({'image':str(image_path),'depth':str(depth_path),'normal':str(normal_path),'mask':str(mask_path),
      'subject_id':f'sample_{sample_id}','label_source':'SynthHuman/DAViD renderer','depth_units':'m','depth_scale_to_m':0.01,
      'normal_convention':'camera_xyz_signed','albedo_color_space':'not_available','albedo':None})
if not manifest_rows: raise RuntimeError('No complete RGB/depth/normal/mask sample groups found.')
MANIFEST.parent.mkdir(parents=True,exist_ok=True)
tmp=MANIFEST.with_suffix('.jsonl.tmp')
tmp.write_text(''.join(json.dumps(r)+'\n' for r in manifest_rows)); tmp.replace(MANIFEST)
print(f'Wrote {len(manifest_rows)} real labeled rows to {MANIFEST}')
print(f'Chunk inventory: RGB={len(RGB_FILES)}, complete={len(manifest_rows)}, incomplete={len(incomplete_samples)}')
if incomplete_samples[:10]: print('First incomplete samples:', incomplete_samples[:10])
print('Available supervision: depth + normals + foreground mask. Albedo: unavailable in SynthHuman.')

### Manifest validation
Provide one JSONL row per labeled sample. In addition to file paths and `subject_id`, each row records `label_source`, `depth_units`, `normal_convention`, and `albedo_color_space` so the targets remain auditable.

In [ ]:
# 4. Read and validate the automatically generated manifest
def load_manifest(path):
    if not path.is_file(): raise FileNotFoundError(f'Manifest missing: {path}. Run the dataset manifest cell first.')
    rows=[json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    required={'image','depth','normal','mask','subject_id','label_source','depth_units','normal_convention'}
    if not rows: raise ValueError('Manifest contains no rows')
    for i,row in enumerate(rows):
        missing=required-row.keys()
        if missing: raise ValueError(f'row {i} missing fields: {sorted(missing)}')
        for key in ('image','depth','normal','mask'):
            p=Path(row[key]); p=p if p.is_absolute() else DATA_ROOT/p
            if not p.is_file(): raise FileNotFoundError(f'row {i} {key}: {p}')
            row[key]=str(p)
        if row.get('albedo'):
            p=Path(row['albedo']); p=p if p.is_absolute() else DATA_ROOT/p
            if not p.is_file(): raise FileNotFoundError(f'row {i} albedo: {p}')
            row['albedo']=str(p)
    return rows
ALL_ROWS=load_manifest(MANIFEST)
PILOT_SAMPLE_COUNT=500
if len(ALL_ROWS)<PILOT_SAMPLE_COUNT:
    raise ValueError(f'Pilot requires {PILOT_SAMPLE_COUNT} complete labeled samples, but manifest has {len(ALL_ROWS)}. Download one full chunk, then rerun manifest generation.')
pilot_rng=random.Random(SEED)
rows=sorted(pilot_rng.sample(ALL_ROWS,PILOT_SAMPLE_COUNT),key=lambda r:r['subject_id'])
PILOT_MANIFEST=DATA_ROOT/'pilot_manifests'/'depth_pilot_500_seed42.jsonl'
PILOT_MANIFEST.parent.mkdir(parents=True,exist_ok=True)
pilot_tmp=PILOT_MANIFEST.with_suffix('.jsonl.tmp'); pilot_tmp.write_text(''.join(json.dumps(r)+'\n' for r in rows)); pilot_tmp.replace(PILOT_MANIFEST)
HAS_ALBEDO=any(r.get('albedo') for r in rows)
print(f'Loaded {len(rows)} deterministic pilot samples from {len(ALL_ROWS)} available rows; labels: depth, normals, mask'+(', albedo' if HAS_ALBEDO else '; albedo not supplied (head will remain untrained)'))
print('Pilot manifest:',PILOT_MANIFEST)

### Identity split
Generate the fixed train/validation/test split before augmentations. The saved `split.json` makes the partition reviewable and reproducible.

In [ ]:
# 5. Split by sample identity (SynthHuman release has no subject-ID field)
def split_by_subject(rows,seed=SEED):
    subjects=sorted({r['subject_id'] for r in rows})
    if len(subjects)<5:
        parts={'train':rows,'val':[],'test':[]}
        print(f'AUDIT MODE: only {len(subjects)} sample(s). This permits the loader audit only; training is blocked until a chunk is downloaded and at least five sample IDs are available.')
        return parts
    rng=random.Random(seed); rng.shuffle(subjects)
    n_test=max(1,round(len(subjects)*TEST_FRACTION)); n_val=max(1,round(len(subjects)*VAL_FRACTION))
    if n_test+n_val>=len(subjects): raise ValueError('Not enough samples for requested split fractions')
    test_ids=set(subjects[:n_test]); val_ids=set(subjects[n_test:n_test+n_val]); train_ids=set(subjects)-test_ids-val_ids
    parts={name:[r for r in rows if r['subject_id'] in ids] for name,ids in [('train',train_ids),('val',val_ids),('test',test_ids)]}
    split_info={'seed':seed,'split_unit':'sample_id (not verified person identity)','train_subjects':sorted(train_ids),'val_subjects':sorted(val_ids),'test_subjects':sorted(test_ids)}
    (OUTPUT_DIR/'split.json').write_text(json.dumps(split_info,indent=2))
    return parts
parts=split_by_subject(rows)
print({k:len(v) for k,v in parts.items()})
print('Evaluation limitation: split is sample-disjoint; the source release does not expose subject IDs for identity-disjoint evaluation.')

### Image and label preprocessing
This cell defines loading/resizing and paired transforms only. It does not download data, change the split, or start training.

In [ ]:
# 6. Read RGB/EXR labels and create a geometry-first dataset
os.environ['OPENCV_IO_ENABLE_OPENEXR']='1'
import cv2

def resize_array(arr,size,is_mask=False):
    arr=np.asarray(arr)
    if arr.ndim==2: arr=arr[...,None]
    mode=Image.Resampling.NEAREST if is_mask or arr.shape[-1]==1 else Image.Resampling.BILINEAR
    planes=[np.asarray(Image.fromarray(arr[...,c].astype(np.float32)).resize((size[1],size[0]),mode),dtype=np.float32) for c in range(arr.shape[-1])]
    return np.stack(planes,axis=-1)

def read_array(path,rgb=False,mask=False,exr_kind=None):
    path=Path(path)
    if path.suffix.lower()=='.exr':
        arr=cv2.imread(str(path),cv2.IMREAD_ANYCOLOR|cv2.IMREAD_ANYDEPTH)
        if arr is None: raise ValueError(f'OpenCV could not read EXR: {path}; check OPENCV_IO_ENABLE_OPENEXR and runtime build')
        arr=arr[...,::-1] if arr.ndim==3 else arr
        if exr_kind=='depth' and arr.ndim==3: arr=arr[...,:1]
    elif path.suffix.lower()=='.npy': arr=np.load(path,allow_pickle=False)
    else:
        with Image.open(path) as im: arr=np.asarray(im.convert('L' if mask else 'RGB'))
    if rgb:
        if arr.dtype!=np.uint8: arr=np.clip(arr*255 if arr.max()<=1 else arr,0,255).astype(np.uint8)
        arr=np.asarray(Image.fromarray(arr).convert('RGB').resize((INPUT_SIZE[1],INPUT_SIZE[0]),Image.Resampling.BILINEAR),dtype=np.float32)/255.
    elif mask:
        arr=resize_array((arr.squeeze()>0).astype(np.uint8)*255,INPUT_SIZE,True)[...,0]>0
    else: arr=resize_array(arr.astype(np.float32),INPUT_SIZE).astype(np.float32)
    return arr

class RelightingDataset(Dataset):
    def __init__(self,rows,augment=False): self.rows,self.augment=rows,augment
    def __len__(self): return len(self.rows)
    def __getitem__(self,idx):
        r=self.rows[idx]
        image=read_array(r['image'],rgb=True)
        depth=read_array(r['depth'],exr_kind='depth')
        normal=read_array(r['normal'],exr_kind='normal')
        mask=read_array(r['mask'],mask=True)
        has_albedo=bool(r.get('albedo'))
        albedo=read_array(r['albedo'],rgb=True) if has_albedo else np.zeros((*INPUT_SIZE,3),np.float32)
        if depth.shape[-1]!=1 or normal.shape[-1]!=3: raise ValueError(f'Bad channels at row {idx}: depth={depth.shape}, normal={normal.shape}')
        raw_depth=depth[...,0].copy()
        # SynthHuman uses 65504 as an invalid EXR depth sentinel (documented by DAViD's visualizer).
        # Filter in source units (cm) before converting to metres; otherwise 655.04 m leaks into training.
        valid_depth=np.isfinite(raw_depth)&(raw_depth>0)&(raw_depth<1000)&(~np.isclose(raw_depth,65504.0,rtol=0.0,atol=1.0))
        depth*=float(r.get('depth_scale_to_m',1.0))
        mask=mask&valid_depth&np.isfinite(normal).all(axis=-1)
        depth=np.nan_to_num(depth,nan=0.,posinf=0.,neginf=0.)
        normal=np.nan_to_num(normal,nan=0.,posinf=0.,neginf=0.)
        if self.augment and random.random()<.5:
            image,depth,normal,albedo,mask=[np.flip(a,axis=1).copy() for a in (image,depth,normal,albedo,mask)]
            normal[...,0]*=-1
        image=(image-np.array([.485,.456,.406],np.float32))/np.array([.229,.224,.225],np.float32)
        tensors=[torch.from_numpy(a).permute(2,0,1).float() for a in (image,depth,normal,albedo)]
        tensors += [torch.from_numpy(mask[None].astype(np.float32)),torch.tensor(float(has_albedo))]
        tensors[2]=F.normalize(tensors[2],dim=0,eps=1e-6)
        tensors[4]*=(torch.linalg.vector_norm(tensors[2],dim=0,keepdim=True)>0).float()
        return tuple(tensors)

### DataLoader and sample audit
Run this after the previous cells. Inspect the displayed example and ranges before constructing the model.

In [ ]:
# 7. Build loaders and audit one real sample
NUM_WORKERS=0  # Drive-mounted EXR files: start reliably; raise only after measuring throughput.
datasets={k:RelightingDataset(v,augment=(k=='train')) for k,v in parts.items()}
loaders={k:DataLoader(v,batch_size=BATCH_SIZE,shuffle=(k=='train'),num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False) for k,v in datasets.items()}
x,d,n,a,m,has_a=datasets['train'][0]
valid_depth_values=d[0][m[0]>0]
print('valid depth metres p01/p50/p99/max',*[float(torch.quantile(valid_depth_values,q)) for q in (0.01,0.50,0.99)],float(valid_depth_values.max()),'normal range',float(n.min()),float(n.max()),'mask fraction',float(m.mean()),'albedo label present',bool(has_a))
assert m.sum()>0,'Empty valid mask'
fig,ax=plt.subplots(1,4,figsize=(15,4)); ax[0].imshow(np.clip(x.permute(1,2,0).numpy()*[.229,.224,.225]+[.485,.456,.406],0,1)); ax[0].set_title('RGB'); ax[1].imshow(d[0],cmap='viridis'); ax[1].set_title('Depth (m)'); ax[2].imshow(n.permute(1,2,0)*.5+.5); ax[2].set_title('Normals'); ax[3].imshow(m[0],cmap='gray'); ax[3].set_title('Valid mask'); [z.axis('off') for z in ax]; plt.show()

In [ ]:
# 8. Shared multi-task architecture with a selectable mobile backbone
# Default preserves the project baseline (FastViT-T8). MobileNetV3-Large is an ablation only.
import timm
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

BACKBONE = 'fastvit_t8.apple_in1k'  # compare candidates via the same train/eval protocol

class FastViTEncoder(nn.Module):
    def __init__(self,pretrained=True):
        super().__init__()
        self.net=timm.create_model('fastvit_t8.apple_in1k',pretrained=pretrained,features_only=True)
        self.channels=tuple(self.net.feature_info.channels())
    def forward(self,x): return self.net(x)

class MobileNetV3Encoder(nn.Module):
    """Experimental alternative; exposes stride-4/8/16/32 features."""
    def __init__(self,pretrained=True):
        super().__init__()
        weights=MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        self.features=mobilenet_v3_large(weights=weights).features
        self.tap_indices=(3,6,12,15)
        self.channels=(24,40,112,160)
    def forward(self,x):
        outputs=[]
        for idx,layer in enumerate(self.features):
            x=layer(x)
            if idx in self.tap_indices: outputs.append(x)
        if len(outputs)!=len(self.channels): raise RuntimeError('Unexpected MobileNetV3 feature taps')
        return outputs

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self,channels):
        super().__init__()
        self.depthwise=nn.Conv2d(channels,channels,3,padding=1,groups=channels,bias=False)
        self.depthwise_bn=nn.BatchNorm2d(channels)
        self.pointwise=nn.Conv2d(channels,channels,1,bias=False)
        self.pointwise_bn=nn.BatchNorm2d(channels)
    def forward(self,x):
        x=F.relu(self.depthwise_bn(self.depthwise(x)),inplace=True)
        return F.relu(self.pointwise_bn(self.pointwise(x)),inplace=True)

class CompactFPN(nn.Module):
    def __init__(self,channels,width=48):
        super().__init__()
        self.lateral=nn.ModuleList([nn.Conv2d(c,width,1) for c in channels])
        self.smooth=nn.ModuleList([DepthwiseSeparableBlock(width) for _ in channels])
    def forward(self,features):
        y=self.smooth[-1](self.lateral[-1](features[-1]))
        for idx in range(len(features)-2,-1,-1):
            y=F.interpolate(y,size=features[idx].shape[-2:],mode='bilinear',align_corners=False)
            y=self.smooth[idx](y+self.lateral[idx](features[idx]))
        return y

class MobileTaskHead(nn.Module):
    def __init__(self,width,out_channels):
        super().__init__(); self.refine=DepthwiseSeparableBlock(width); self.predict=nn.Conv2d(width,out_channels,1)
    def forward(self,x,out_size):
        return F.interpolate(self.predict(self.refine(x)),size=out_size,mode='bilinear',align_corners=False)

class PortraitRelightingModel(nn.Module):
    """One RGB crop -> depth, unit normals, and diffuse albedo."""
    def __init__(self,backbone=BACKBONE,pretrained=True,fpn_width=48):
        super().__init__()
        if backbone=='fastvit_t8.apple_in1k': self.encoder=FastViTEncoder(pretrained=pretrained)
        elif backbone=='mobilenet_v3_large': self.encoder=MobileNetV3Encoder(pretrained=pretrained)
        else: raise ValueError(f'Unsupported backbone: {backbone}')
        self.backbone_name=backbone
        self.neck=CompactFPN(self.encoder.channels,width=fpn_width)
        self.depth_head=MobileTaskHead(fpn_width,1)
        self.normal_head=MobileTaskHead(fpn_width,3)
        self.albedo_head=MobileTaskHead(fpn_width,3)
    def forward(self,x):
        out_size=x.shape[-2:]; fused=self.neck(self.encoder(x))
        depth=F.softplus(self.depth_head(fused,out_size))
        normals=F.normalize(self.normal_head(fused,out_size),p=2,dim=1,eps=1e-6)
        albedo=torch.sigmoid(self.albedo_head(fused,out_size))
        return depth,normals,albedo

model=PortraitRelightingModel(backbone=BACKBONE,pretrained=True).to(DEVICE)
nparams=sum(p.numel() for p in model.parameters())
with torch.no_grad(): outputs=model(torch.zeros(2,3,*INPUT_SIZE,device=DEVICE))
assert [tuple(t.shape) for t in outputs]==[(2,1,*INPUT_SIZE),(2,3,*INPUT_SIZE),(2,3,*INPUT_SIZE)]
assert all(torch.isfinite(t).all() for t in outputs)
print(f'Backbone={BACKBONE}; total parameters={nparams/1e6:.2f}M; FPN width=48; feature channels={model.encoder.channels}')


In [ ]:
# 9. Masked task losses and metrics; absent albedo labels are excluded, never replaced by RGB
class MultiTaskObjective(nn.Module):
    def __init__(self): super().__init__(); self.log_vars=nn.Parameter(torch.zeros(3))
    @staticmethod
    def masked_mean(value,mask): return (value*mask).sum()/mask.sum().clamp_min(1.)
    def forward(self,pd,td,pn,tn,pa,ta,mask,has_albedo):
        if DEPTH_MODE=='relative':
            pd0=pd.clamp_min(1e-5).log(); td0=td.clamp_min(1e-5).log()
            for b in range(pd.shape[0]):
                valid=mask[b,0].bool()
                if not valid.any(): raise ValueError('A sample has an empty valid mask')
                pd0[b]-=torch.median(pd0[b,0][valid]); td0[b]-=torch.median(td0[b,0][valid])
            ld=self.masked_mean(F.smooth_l1_loss(pd0,td0,reduction='none'),mask)
        else:
            ld=self.masked_mean(F.smooth_l1_loss(pd,td,reduction='none'),mask)
        cos=(F.normalize(pn,dim=1,eps=1e-6)*F.normalize(tn,dim=1,eps=1e-6)).sum(1,keepdim=True).clamp(-1,1)
        ln=self.masked_mean(1-cos,mask)
        losses=[ld,ln]; names=['depth_loss','normal_loss']
        if has_albedo.bool().any():
            amask=mask*has_albedo[:,None,None,None]
            la=self.masked_mean(torch.abs(pa-ta),amask.expand_as(pa)); losses.append(la); names.append('albedo_loss')
        idx=[0,1]+([2] if len(losses)==3 else [])
        total=sum(torch.exp(-self.log_vars[j])*v+self.log_vars[j] for j,v in zip(idx,losses))
        return total,{name:value.detach() for name,value in zip(names,losses)}

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); sums={}; counts={}
    for x,d,n,a,m,has_a in loader:
        x,d,n,a,m,has_a=[z.to(DEVICE,non_blocking=True) for z in (x,d,n,a,m,has_a)]
        pd,pn,pa=model(x); valid=m[:,0].bool()
        dv=(pd[:,0]-d[:,0])[valid]
        result={'depth_mae_m':dv.abs().mean(),'depth_rmse_m':dv.square().mean().sqrt()}
        p_rel=pd[:,0].clamp_min(1e-5).log(); t_rel=d[:,0].clamp_min(1e-5).log()
        for b in range(x.shape[0]):
            mb=valid[b]
            if not mb.any(): raise ValueError('Evaluation sample has empty mask')
            p_rel[b]-=torch.median(p_rel[b][mb]); t_rel[b]-=torch.median(t_rel[b][mb])
        rr=(p_rel-t_rel)[valid]; result['depth_log_rmse_median_aligned']=rr.square().mean().sqrt()
        dot=(F.normalize(pn,dim=1)*F.normalize(n,dim=1)).sum(1).clamp(-1,1); ang=torch.rad2deg(torch.acos(dot[valid]))
        result.update({'normal_mean_deg':ang.mean(),'normal_median_deg':ang.median(),'normal_within_11_25_deg':(ang<11.25).float().mean()})
        if has_a.bool().any():
            valid_a=valid&has_a[:,None,None].bool(); av=(pa-a).permute(0,2,3,1)[valid_a]
            result['albedo_mae']=av.abs().mean()
        for k,v in result.items(): sums[k]=sums.get(k,0.)+float(v); counts[k]=counts.get(k,0)+1
    if not counts: raise ValueError('Evaluation partition is empty')
    return {k:sums[k]/counts[k] for k in sums}

objective=MultiTaskObjective().to(DEVICE)
optimizer=torch.optim.AdamW(list(model.parameters())+list(objective.parameters()),lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=.5,patience=3)
scaler=torch.amp.GradScaler(device='cuda',enabled=(DEVICE.type=='cuda'))
print('Active supervised heads:', 'depth + normals + albedo' if HAS_ALBEDO else 'depth + normals; albedo head is not trained')

In [ ]:
# 10. Timed training with resumable mid-epoch checkpoints on Drive
CHECKPOINT=OUTPUT_DIR/'best_checkpoint.pt'
LAST=OUTPUT_DIR/'last_checkpoint.pt'
HISTORY_PATH=OUTPUT_DIR/'history.json'
PROGRESS_PATH=OUTPUT_DIR/'progress.json'
CHECKPOINT_EVERY_MINUTES=10
PROGRESS_EVERY_STEPS=25  # Drive progress log interval; the live bar updates every batch
print('Persistent checkpoints/history location:', OUTPUT_DIR)
partition_sizes={name:len(group) for name,group in parts.items()}
if any(size==0 for size in partition_sizes.values()):
    raise RuntimeError(f'Cannot train: train/validation/test split is incomplete: {partition_sizes}. Download a full chunk and rerun manifest/split/loader cells first.')

def save_checkpoint(payload,path):
    temp_path=path.with_name(path.name+'.tmp')
    torch.save(payload,temp_path)
    temp_path.replace(path)

def checkpoint_state(epoch,batch_in_epoch,epoch_order,score_value,train_totals=None,task_totals=None,train_seconds_elapsed=0.0):
    return {'epoch':epoch,'batch_in_epoch':batch_in_epoch,'epoch_order':epoch_order,'partial_train_totals':train_totals or [],'partial_task_totals':task_totals or {},'train_seconds_elapsed':train_seconds_elapsed,'model':model.state_dict(),'objective':objective.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'best_score':score_value,'history':history,'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,'config':{'seed':SEED,'input_size':INPUT_SIZE,'depth_mode':DEPTH_MODE,'backbone':BACKBONE,'fpn_width':48,'albedo_supervised':HAS_ALBEDO}}

history=[]; start_epoch=0; resume_batch=0; resume_order=None; best_score=float('inf'); resume_train_totals=[]; resume_task_totals={}; resume_train_seconds=0.0
if LAST.exists():
    ckpt=torch.load(LAST,map_location=DEVICE,weights_only=False)
    model.load_state_dict(ckpt['model']); objective.load_state_dict(ckpt['objective']); optimizer.load_state_dict(ckpt['optimizer']); scheduler.load_state_dict(ckpt['scheduler']); scaler.load_state_dict(ckpt['scaler'])
    start_epoch=ckpt['epoch']; resume_batch=ckpt.get('batch_in_epoch',0); resume_order=ckpt.get('epoch_order'); best_score=ckpt.get('best_score',float('inf')); history=ckpt.get('history',[]); resume_train_totals=ckpt.get('partial_train_totals',[]); resume_task_totals=ckpt.get('partial_task_totals',{}); resume_train_seconds=ckpt.get('train_seconds_elapsed',0.0)
    random.setstate(ckpt['python_rng']); np.random.set_state(ckpt['numpy_rng']); torch.set_rng_state(ckpt['torch_rng'])
    if torch.cuda.is_available() and ckpt.get('cuda_rng') is not None: torch.cuda.set_rng_state_all(ckpt['cuda_rng'])
    print(f'Resuming epoch {start_epoch+1}, after {resume_batch} completed batches.')

for epoch in range(start_epoch,EPOCHS):
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    epoch_start=time.perf_counter()
    model.train(); objective.train()
    if epoch==start_epoch and resume_batch>0:
        train_totals=list(resume_train_totals); task_totals={k:list(v) for k,v in resume_task_totals.items()}
    else:
        train_totals=[]; task_totals={'depth_loss':[],'normal_loss':[]}
        if HAS_ALBEDO: task_totals['albedo_loss']=[]
    if epoch==start_epoch and resume_order is not None:
        epoch_order=resume_order
    else:
        epoch_order=torch.randperm(len(datasets['train']),generator=torch.Generator().manual_seed(SEED+epoch)).tolist()
        resume_batch=0
    start_sample=resume_batch*BATCH_SIZE
    remaining_indices=epoch_order[start_sample:]
    epoch_loader=DataLoader(datasets['train'],batch_size=BATCH_SIZE,sampler=remaining_indices,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False,generator=torch.Generator().manual_seed(SEED+10000+epoch))
    prior_train_seconds=resume_train_seconds if epoch==start_epoch else 0.0
    last_checkpoint_time=time.monotonic()
    from tqdm.auto import tqdm
    full_batches=(len(datasets['train'])+BATCH_SIZE-1)//BATCH_SIZE
    progress_bar=tqdm(epoch_loader,total=full_batches,initial=resume_batch,desc=f'Epoch {epoch+1}/{EPOCHS}',unit='batch',dynamic_ncols=True)
    for batch_idx,(x,d,n,a,m,has_a) in enumerate(progress_bar,start=resume_batch):
        x,d,n,a,m,has_a=[z.to(DEVICE,non_blocking=True) for z in (x,d,n,a,m,has_a)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,dtype=(torch.float16 if DEVICE.type=='cuda' else torch.bfloat16),enabled=(DEVICE.type=='cuda')):
            pd,pn,pa=model(x); loss,parts=objective(pd,d,pn,n,pa,a,m,has_a)
        if not torch.isfinite(loss): raise FloatingPointError(f'Non-finite training loss at epoch {epoch}')
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(objective.parameters()),1.0); scaler.step(optimizer); scaler.update()
        train_totals.append(float(loss.detach()))
        for k,v in parts.items(): task_totals[k].append(float(v))
        completed_batches=batch_idx+1
        progress_bar.set_postfix(loss=f'{float(loss.detach()):.4f}',depth=f'{float(parts.get("depth_loss",torch.tensor(float("nan")))):.4f}',normal=f'{float(parts.get("normal_loss",torch.tensor(float("nan")))):.4f}',refresh=False)
        if completed_batches%PROGRESS_EVERY_STEPS==0:
            progress={'epoch':epoch+1,'epochs_total':EPOCHS,'batches_complete':completed_batches,'batches_total':(len(datasets['train'])+BATCH_SIZE-1)//BATCH_SIZE,'latest_train_loss':float(loss.detach()),'last_update_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
            progress_tmp=PROGRESS_PATH.with_name(PROGRESS_PATH.name+'.tmp'); progress_tmp.write_text(json.dumps(progress,indent=2)); progress_tmp.replace(PROGRESS_PATH)
            print(f"epoch {epoch+1:03d}/{EPOCHS} batch {completed_batches}/{progress['batches_total']} loss={float(loss.detach()):.4f}")
        if time.monotonic()-last_checkpoint_time>=CHECKPOINT_EVERY_MINUTES*60:
            if DEVICE.type=='cuda': torch.cuda.synchronize()
            save_checkpoint(checkpoint_state(epoch,completed_batches,epoch_order,best_score,train_totals,task_totals,prior_train_seconds+time.perf_counter()-epoch_start),LAST)
            last_checkpoint_time=time.monotonic()
            print(f'Saved resumable checkpoint after epoch {epoch+1}, batch {completed_batches}.')
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    train_seconds=prior_train_seconds+time.perf_counter()-epoch_start
    val_start=time.perf_counter()
    val_metrics=evaluate(model,loaders['val'])
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    val_seconds=time.perf_counter()-val_start
    epoch_seconds=time.perf_counter()-epoch_start
    samples_per_second=len(datasets['train'])/max(train_seconds,1e-6)
    eta_seconds=max(EPOCHS-epoch-1,0)*epoch_seconds
    score=(val_metrics['depth_rmse_m']/0.1)+(val_metrics['normal_mean_deg']/30.0)+(val_metrics.get('albedo_mae',0.0)/0.1)
    scheduler.step(score)
    record={'epoch':epoch+1,'train_loss':float(np.mean(train_totals)),**{k:float(np.mean(v)) for k,v in task_totals.items()},'val':val_metrics,'selection_score':score,'train_seconds':train_seconds,'validation_seconds':val_seconds,'epoch_seconds':epoch_seconds,'train_samples_per_second':samples_per_second,'eta_seconds':eta_seconds,'lr':optimizer.param_groups[0]['lr'],'task_sigmas':torch.exp(objective.log_vars.detach()).cpu().tolist()}
    history.append(record)
    history_tmp=HISTORY_PATH.with_name(HISTORY_PATH.name+'.tmp'); history_tmp.write_text(json.dumps(history,indent=2)); history_tmp.replace(HISTORY_PATH)
    completed_state=checkpoint_state(epoch+1,0,None,min(best_score,score),[],{},0.0)
    save_checkpoint(completed_state,LAST)
    if score<best_score:
        best_score=score; completed_state['best_score']=best_score; completed_state['best_epoch']=epoch+1; save_checkpoint(completed_state,CHECKPOINT)
    progress={'epoch':epoch+1,'epochs_total':EPOCHS,'status':'epoch_complete','train_seconds':train_seconds,'validation_seconds':val_seconds,'epoch_seconds':epoch_seconds,'train_samples_per_second':samples_per_second,'eta_seconds':eta_seconds}
    progress_tmp=PROGRESS_PATH.with_name(PROGRESS_PATH.name+'.tmp'); progress_tmp.write_text(json.dumps(progress,indent=2)); progress_tmp.replace(PROGRESS_PATH)
    resume_batch=0; resume_order=None; resume_train_totals=[]; resume_task_totals={}; resume_train_seconds=0.0
    print(f"epoch {epoch+1:03d}/{EPOCHS} train={record['train_loss']:.4f} val depth(RMSE m)={val_metrics['depth_rmse_m']:.4f} normal(mean°)={val_metrics['normal_mean_deg']:.2f} albedo(MAE)={val_metrics.get('albedo_mae', float('nan')):.4f} score={score:.4f} train={train_seconds/60:.1f}m val={val_seconds/60:.1f}m rate={samples_per_second:.1f} samples/s ETA={eta_seconds/60:.1f}m")

if not CHECKPOINT.exists() and LAST.exists():
    save_checkpoint(torch.load(LAST,map_location=DEVICE,weights_only=False),CHECKPOINT)
if not CHECKPOINT.exists(): raise RuntimeError('No checkpoint found. Complete at least one epoch first.')
best=torch.load(CHECKPOINT,map_location=DEVICE,weights_only=False); model.load_state_dict(best['model'])
print('Best checkpoint epoch:',best.get('best_epoch',best['epoch']),'selection score:',best['best_score'])

In [ ]:
# 11. Locked test evaluation and qualitative inspection (after model selection)
test_metrics=evaluate(model,loaders['test'])
print(json.dumps(test_metrics,indent=2))
metrics_path=OUTPUT_DIR/'test_metrics.json'; metrics_tmp=metrics_path.with_name(metrics_path.name+'.tmp')
metrics_tmp.write_text(json.dumps(test_metrics,indent=2)); metrics_tmp.replace(metrics_path)
# PSNR/SSIM/LPIPS for albedo and target-light RGB only make sense against independent reference captures.
model.eval(); x,d,n,a,m,has_a=next(iter(loaders['test'])); x=x.to(DEVICE)
with torch.inference_mode(): pd,pn,pa=model(x)
rgb=((x[0].cpu()*torch.tensor([.229,.224,.225])[:,None,None])+torch.tensor([.485,.456,.406])[:,None,None]).permute(1,2,0).clamp(0,1)
fig,ax=plt.subplots(2,4,figsize=(14,7)); ax[0,0].imshow(rgb); ax[0,0].set_title('Input'); ax[0,1].imshow(d[0,0],cmap='viridis'); ax[0,1].set_title('Depth GT'); ax[0,2].imshow(n[0].permute(1,2,0)*.5+.5); ax[0,2].set_title('Normals GT'); ax[0,3].imshow(a[0].permute(1,2,0) if bool(has_a[0]) else np.zeros((INPUT_SIZE[0],INPUT_SIZE[1],3))); ax[0,3].set_title('Albedo GT' if bool(has_a[0]) else 'Albedo label absent'); ax[1,0].imshow(pd[0,0].cpu(),cmap='viridis'); ax[1,0].set_title('Depth pred'); ax[1,1].imshow(pn[0].cpu().permute(1,2,0)*.5+.5); ax[1,1].set_title('Normals pred'); ax[1,2].imshow(pa[0].cpu().permute(1,2,0)); ax[1,2].set_title('Albedo output (untrained)' if not HAS_ALBEDO else 'Albedo pred'); ax[1,3].imshow(m[0,0],cmap='gray'); ax[1,3].set_title('Valid mask'); [z.axis('off') for z in ax.flat]; plt.tight_layout(); plt.show()

In [ ]:
# 12. Optional fixed-shape Core ML export; run with a pinned coremltools version.
# Conversion success is not a performance result. Follow with Xcode performance report on target iPhone(s).
EXPORT_COREML=False
if EXPORT_COREML and not HAS_ALBEDO:
    raise RuntimeError('Albedo has no training labels in the current SynthHuman trial; do not export this as a trained three-output model.')
if EXPORT_COREML:
    import coremltools as ct
    class ExportWrapper(nn.Module):
        def __init__(self,base): super().__init__(); self.base=base
        def forward(self,x): return self.base(x)
    export_model=ExportWrapper(model.eval()).cpu().eval()
    example=torch.zeros(1,3,*INPUT_SIZE)
    traced=torch.jit.trace(export_model,example,strict=True)
    # The model takes ImageNet-normalized RGB tensors; export as tensor input and normalize in app preprocessing.
    mlmodel=ct.convert(traced,source='pytorch',inputs=[ct.TensorType(name='image',shape=example.shape)],outputs=[ct.TensorType(name='depth'),ct.TensorType(name='normals'),ct.TensorType(name='albedo')],convert_to='mlprogram',minimum_deployment_target=ct.target.iOS16,compute_precision=ct.precision.FLOAT16)
    mlmodel.author='Apple Capstone Portrait Relighting'; mlmodel.short_description='Multi-task portrait geometry and reflectance research model; evaluate on device before use.'
    out=OUTPUT_DIR/'portrait_relighting.mlpackage'; mlmodel.save(str(out)); print('Saved',out)
    # Numerical parity check on a representative fixed-shape tensor.
    if platform.system() == 'Darwin':
        with torch.inference_mode(): torch_out=export_model(example)
        coreml_out=mlmodel.predict({'image':example.numpy()})
        for name,ref in zip(('depth','normals','albedo'),torch_out):
            delta=np.abs(coreml_out[name]-ref.numpy())
            print(name,'max_abs_error',float(delta.max()),'mean_abs_error',float(delta.mean()))
            if not np.isfinite(delta).all(): raise FloatingPointError(f'Non-finite Core ML output for {name}')
    else:
        print('Export completed. Core ML prediction/parity check requires macOS; run that check on a Mac or target device.')

# Device evaluation checklist:
# 1) Xcode Model Performance Report: operation placement (ANE/GPU/CPU), load and prediction duration.
# 2) On each target iPhone/iOS: warm-up; >=200 predictions; report p50/p95, cold load, package size, peak memory.
# 3) Measure Vision segmentation + preprocessing + model + relighting + compositing frame time; include thermal steady state.
# 4) Compare FP32/FP16/quantized candidates for parity, quality, model size, and device latency.
# 5) 60 FPS requires the complete pipeline to sustain <16.7 ms/frame with acceptable quality and thermal behavior.

## Interpretation and required experiments
## Backbone evidence and decision
- **Keep FastViT-T8 as the baseline.** Apple reports FastViT-T8 at 3.6M parameters and provides Core ML classification measurements on iPhone models (for example 0.52 ms on iPhone 16 Pro). The FastViT paper and official model page support mobile efficiency and Core ML availability. Those figures are for the backbone/classifier model, not our 256×256 three-output dense model.
- **MobileNetV3-Large is a useful ablation.** Its paper targets mobile CPU efficiency, reports about 5.4M parameters, and introduces LR-ASPP for dense prediction. Its published CPU measurements use Pixel devices; they do not prove better Apple Neural Engine performance.
- **These published latency values are not directly comparable.** The devices, runtime, input graph, precision, and task differ. Change only the backbone while holding the dataset split, FPN/heads, resolution, training schedule, and Core ML conversion constant. Select based on held-out task quality, model size, and Xcode measurements on the intended iPhone.

- First establish that label overlays and coordinate conventions are correct. Pseudo-label metrics quantify agreement with the teacher, not necessarily physical truth.
- Ablate: (A) FastViT-T8 baseline with compact FPN, (B) MobileNetV3-Large comparison using the same FPN/heads, (C) fixed versus uncertainty-weighted losses. Change only the backbone for the backbone ablation; compare validation metrics, parameter count, converted model size, and on-device latency. Published backbone-only classification timings are not comparable to this dense prediction graph. - Compare against task-appropriate baselines: teacher outputs for geometry, constant/median depth where meaningful, and input-copy albedo.
- For the product claim (relighting), build a controlled capture set with fixed subject, camera/exposure/white balance, and measured light positions/intensities. Hold out identities and lighting configurations. Evaluate relit RGB against captures and inspect temporal stability on video.
- References: [FastViT paper](https://arxiv.org/abs/2303.14189); [Sapiens paper](https://arxiv.org/abs/2408.12569); [uncertainty weighting](https://arxiv.org/abs/1705.07115); [Core ML performance report](https://developer.apple.com/documentation/coreml/analyzing-a-core-ml-model-s-performance-in-xcode); [Core ML Tools PyTorch conversion](https://apple.github.io/coremltools/docs-guides/source/convert-pytorch.html); [MobileNetV3 paper](https://openaccess.thecvf.com/content_ICCV_2019/html/Howard_Searching_for_MobileNetV3_ICCV_2019_paper.html).
## Dataset trial scope
- The automatic Colab trial uses the official Microsoft SynthHuman/DAViD download utility. Its synthetic sample labels include RGB, absolute depth in centimetres, XYZ normals, and alpha masks; it does not include albedo. The first geometry trial therefore supervises depth and normals only. Do not interpret the untrained albedo output as a result.
- The official release offers a single-sample smoke test and a single chunk (up to 5,000 samples; up to 8.75 GB); the full dataset is about 330 GB. The trial split uses sample IDs because the distributed files do not expose identity IDs, so it is not an identity-disjoint generalization test.
